# 5회차 보충 · 지시문으로 모델 다루기(프롬프트 샌드박스)

11/20 수업(5회차) 내용을 **혼자서 다시 해 볼 수 있게** 만든 개인 실습임. 팀 서버 없이 내 서버에서 30~40분이면 끝남.
6회차 노트북보다 먼저 해 두면 좋지만, 순서를 바꿔도 되고 필요한 단계만 골라 해도 됨.

**전체 흐름부터.** 언어 모델은 받은 글 뒤에 이어질 글을 만들어 내는 프로그램임. 그래서 모델을 고치지 않아도 **받는 글(입력)** 을 바꾸면 답이 바뀜.
입력에는 세 가지가 들어감 : 지시문(역할과 규칙), 예시(이렇게 답하라는 본보기), 질문. 여기에 답을 만들 때의 설정(temperature, 최대 길이)이 더해짐.

`지시문 + 예시 + 질문` → (Chat Template 으로 한 줄의 글이 됨) → 모델 → 답 → 확인 질문으로 점수를 잼

- 5회차 : 입력을 바꿔 답을 바꿈(오늘 이 노트북)
- 6회차 : 입력에 **내 문서**를 더 붙임(RAG)
- 7회차 : 입력이 아니라 **모델 자체**를 조금 고침(LoRA)

| 단계 | 하는 일 | 필수 |
|---|---|---|
| 0 · 1 | 시작, 모델 올리기 | 필수 |
| 2 | 지시문 써 보기 | 필수 |
| 3 | 예시 넣어 보기, 모델이 실제로 받는 글 보기 | 필수 |
| 4 | 생성 설정(temperature, 최대 길이) | 필수 |
| 5 | 형식 지키게 하기 | 선택 |
| 6 | 문제 입력 넣어 보기 | 선택 |
| 7 | 내 확인 질문과 기준 점수 | 필수 |
| 8 | 판단 기록 · 결과 묶기 | 필수 |

## 0 · 1. 시작과 모델 올리기
아래 두 셀을 차례로 실행함. 기본 모델 `skt/A.X-4.0-Light`를 올리면 됨(약 30초). 모델을 올린다는 것은 가중치 파일을 GPU 메모리에 싣는 일이고, 계기판의 막대가 그만큼 참.

In [ ]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "lab.py").exists())
sys.path.insert(0, str(root / "src"))
import lab
lab.start(5)

In [ ]:
lab.model_panel()

## 2. 지시문 써 보기
질문은 그대로 두고 **지시문**만 줬을 때 답이 어떻게 달라지는지 나란히 봄. 지시문에는 역할, 말투, 길이, 지켜야 할 규칙을 적음.

**왜 하나** : 실제 서비스에서 사용자는 질문만 보냄. 답의 말투와 범위를 정하는 것은 개발자가 미리 넣어 두는 지시문임. 같은 모델이 지시문 하나로 전혀 다른 챗봇이 되는 것을 먼저 눈으로 확인함.

| 바꿔 볼 것 | 예 | 볼 것 |
|---|---|---|
| 길이 | 두 문장 이내로 답한다 | 답이 실제로 짧아지는가 |
| 역할 | 너는 초등학생을 가르치는 선생님이다 | 낱말과 예시가 쉬워지는가 |
| 금지 규칙 | 모르는 내용은 모른다고 답한다 | 지어내던 답이 줄어드는가 |

마음에 드는 지시문이 나오면 '지시문을 내 설정으로 저장'을 켜고 다시 누름. 6회차 챗봇에서도 같은 지시문을 쓰게 됨.

In [ ]:
lab.prompt_panel()

## 3. 예시 넣어 보기
말로 설명하기 어려운 것은 **예시**로 보여 주는 편이 빠름. 예시 입력란에 `질문 => 원하는 답`을 두세 줄 적고 다시 물어봄(Few-shot 이라고 부름).

**왜 하나** : "친절하게 답해"라고 쓰는 것보다 친절한 답 두 개를 보여 주는 것이 더 잘 통하는 경우가 많음. 대신 예시만큼 입력이 길어져서 응답이 느려지고 비용이 늘어남. 아래 작은 글씨의 입력 토큰 수를 같이 봄.

- '모델에 실제로 들어가는 글 보기'를 켜면 지시문 · 예시 · 질문이 특수 기호와 함께 한 개의 긴 글로 이어진 모습이 보임. 이렇게 이어 붙이는 규칙이 **Chat Template** 임. 모델마다 기호가 달라서, 이 형식을 틀리게 넣으면 같은 모델도 답이 나빠짐
- 예시를 0건, 1건, 3건으로 늘려가며 답이 예시를 얼마나 따라가는지 봄. 예시의 말버릇까지 그대로 따라 하는 지도 확인함

In [ ]:
lab.prompt_panel()

## 4. 생성 설정
같은 질문을 여러 번 생성해 답이 얼마나 달라지는지 봄. 맨 위 '기준' 줄은 temperature 0 으로 만든 답임.

**왜 하나** : 모델은 다음에 올 낱말을 확률로 고름. **temperature** 는 확률이 낮은 낱말을 얼마나 허용할지 정하는 값임. 0 이면 항상 가장 확률이 높은 낱말만 골라서 몇 번을 돌려도 같은 답이 나오고, 높이면 답이 매번 달라짐. 그래서 점수를 재는 실험에서는 temperature 를 0 으로 둠(답이 매번 바뀌면 설정을 바꾼 효과인지 우연인지 알수 없음).

| 바꿔 볼 것 | 값 | 볼 것 |
|---|---|---|
| temperature | 0 → 0.7 → 1.5 | '서로 다른 답' 수, 1.5 에서 문장이 이상해지는가 |
| 최대 길이 | 20 → 100 → 400 | '길이 제한에 걸림'이 '예'인 답은 문장이 중간에 끊김 |
| 질문 종류 | 사실 질문 대 이름 지어 줘 같은 창작 질문 | 어느 질문이 temperature 에 더 민감한가 |

In [ ]:
lab.gen_panel()

## 5. 형식 지키게 하기(선택)
답을 프로그램이 받아서 쓰려면 형식이 일정해야 함. 형식(JSON, 한 단어, 세 줄)을 지시했을 때 실제로 지키는 비율을 잼.

**왜 하나** : 사람은 답이 조금 달라도 읽을 수 있지만 프로그램은 `{"category": ...}` 꼴이 아니면 오류가 남. 8회차 이후 Agent 는 모델의 답을 프로그램이 받아 처리하므로 형식 준수율이 곧 서비스가 멈추지 않는 비율임.

- 먼저 그대로 실행해 '지시 없음'과 '형식 지시'의 준수율을 비교함
- 준수율이 1.0 이 안되면 예시 입력란에 형식을 지킨 예시를 한두 줄 넣고 다시 잼
- 한 단어 형식에서 목록에 없는 단어를 답하는 질문이 어떤 질문인지 읽어 봄

In [ ]:
lab.format_panel()

## 6. 문제 입력 넣어 보기(선택)
정상 질문 하나를 여섯 가지로 바꿔 넣음 : 오타, 지시를 무시하라는 요구, 관계없는 질문, 영어, 정보가 부족한 질문.

**왜 하나** : 사용자는 개발자가 예상한 대로 묻지 않음. 지시문이 정상 질문에서만 통하는지, 이상한 입력에서도 버티는지 미리 넣어 보는 것임. '지시 무시 요구' 줄의 자동 확인이 '지시가 무시됨'으로 나오면 지시문을 고쳐야 함.

- 지시문을 비운 채 한 번, 2단계에서 만든 지시문을 넣고 한 번 실행해 비교함
- 지시문에 "범위를 벗어난 질문에는 안내할 수 없다고 답한다" 같은 규칙을 넣으면 '관계없는 질문' 줄이 어떻게 바뀌는지 봄

In [ ]:
lab.probe_panel()

## 7. 내 확인 질문과 기준 점수
내가 만들 챗봇이 답할수 있어야 하는 질문을 5건 이상 적음. 한 줄에 `질문 | 답에 꼭 들어가야 할 단어` 형식임.
'저장하고 풀기'를 누르면 지시가 없을 때(기준)와 내 지시문을 줬을 때의 점수를 비교함.

**왜 하나** : 좋아졌는지 말하려면 먼저 **기준 점수(Baseline)** 가 있어야 함. 지시문을 고칠 때마다 같은 질문으로 다시 재야 "느낌상 좋아진 것 같음"이 아니라 숫자로 말할 수 있음. 6회차와 7회차도 같은 확인 질문으로 이어서 잼.

- 모델이 원래 모르는 내용(우리 동아리 회비, 내가 지어낸 가게 규정)은 지시문만으로는 점수가 잘 안 오름. 지시문에 그 사실을 직접 적어 주면 오름. 사실이 많아지면 지시문에 다 적을 수 없으므로 6회차의 문서 검색(RAG)이 필요해짐

In [ ]:
lab.prompt_score_panel()

## 8. 판단 기록 · 결과 묶기
어떤 지시문과 설정으로 정했는지, 왜 그렇게 정했는지 적음. 숫자(점수, 준수율, 토큰 수)를 두 개 이상 넣어 적음.

In [ ]:
lab.note_panel()

In [ ]:
lab.pack()

---
### 예시 과제(하나를 골라 끝까지 해 봄)
1. **우리 동아리 안내 담당** : 지시문에 회비 · 모임 시간 · 가입 방법을 적어 주고, 확인 질문 5건의 점수를 기준보다 올림. 반말로 물어도 존댓말로 답하게 함
2. **문의 분류기** : 5단계에서 한 단어 형식(문의, 불만, 칭찬, 기타)의 준수율을 1.0 으로 만듦. 예시는 3건까지만 씀
3. **세 줄 요약기** : 긴 글을 붙여 넣으면 정확히 세 줄로 요약하게 함. 최대 길이를 얼마로 두어야 끊기지 않는지 찾음

### 도전 과제
- 지시문 길이를 절반으로 줄이고도 같은 점수가 나오게 고쳐 봄(입력 토큰이 곧 비용임)
- 6단계의 '지시 무시 요구'를 내 말로 더 교묘하게 바꿔 넣어 보고, 그래도 버티는 지시문을 만듦
- `Qwen3-0.6B` 를 하나 더 올려 같은 지시문을 줌. 작은 모델은 어떤 지시를 먼저 놓치는가 : `lab.load("Qwen/Qwen3-0.6B")` 뒤 다시 실행, 끝나면 `lab.use("A.X-4.0-Light")`
- 조절판 없이 직접 부르기 : `lab.prompt_try("질문", system="지시문")`, `lab.sample("질문", temperature=1.2, n=6)`, `lab.format_try("json", "name, price")`, `lab.probe("질문")`